In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [3]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("CK1/data/ck1_train_1")

X2_all = load_datasets("CK1/data/ck1_val_1")

X3_all = load_datasets("CK1/data/ck1_test_1")

#transformer = ChemBERTaTransformer()
#X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
#X_val_emb = transformer.transform(X2_all.df["Drug"])
#X_test_emb = transformer.transform(X3_all.df["Drug"])

In [5]:
display(X2_all.df)

,QSPRID,Y,Drug,Y_original
QSPRID,,,,
A2ARDataset_000,A2ARDataset_000,False,COc1cc(C)c(NC(=O)CN(C)C)cc1Nc1nc(Nc2cccc(F)c2C...,False
A2ARDataset_001,A2ARDataset_001,False,CC(C)(C)c1cc(NC(=O)C(=O)c2cccc3ccccc23)n(-c2cc...,False
A2ARDataset_002,A2ARDataset_002,False,Nc1nnc(-c2cc3c(Oc4ccc(Cl)cc4)cncc3s2)o1,False
A2ARDataset_003,A2ARDataset_003,False,OCc1ccc(-c2nc(-c3ccccn3)c(-c3ccc4c(c3)OCO4)[nH...,False
A2ARDataset_004,A2ARDataset_004,False,CS(=O)(=O)CCNCc1ccoc1-c1ccc2ncnc(Nc3ccc(OCc4cc...,False
...,...,...,...,...
A2ARDataset_150,A2ARDataset_150,False,COc1cc2nc3nc(-c4cccs4)c(-c4cccs4)nc3nc2cc1OC,False
A2ARDataset_151,A2ARDataset_151,False,NC1(C(=O)NCc2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1,False
A2ARDataset_152,A2ARDataset_152,False,CCc1cccc(NC(=O)Nc2ccc(Oc3ccc4nc(NC(=O)OC)[nH]c...,False


In [6]:
from MolEval import MolEmb 
model_name = 'RoBERTa_ZINC'  # Replace with the model you want to use

X1_all.df["SMILES"] = X1_all.df["Drug"]
extractor = MolEmb.EmbeddingExtractor(model_name=model_name, df=X1_all.df)
emb, X1_all.df = extractor.get_embeddings()
print(emb)


No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'
Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.p

          0         1         2         3         4         5         6    \
0    0.839525  0.657080  0.260560 -0.284325 -0.212575 -0.259249 -0.536546   
1    0.521116  0.330557  0.510996 -0.643829 -0.374516 -0.092652  0.079537   
2    0.406173  0.094941  0.932744 -1.015941 -0.701294  0.265186 -0.147850   
3    0.181259  0.227875  0.667421 -0.794918 -0.201879  0.051985  0.281510   
4    0.012582  0.105825  0.637197 -0.452851 -0.308520 -0.366891 -0.377147   
..        ...       ...       ...       ...       ...       ...       ...   
483  0.908416  0.707765  0.471640 -0.101745 -0.161899 -0.017486  0.037784   
484  0.649540  0.024394  0.535391 -0.161667 -0.518024 -0.034655  0.029912   
485  0.412816  0.103349  0.281799 -0.509588 -0.296525 -0.247597  0.082339   
486  0.245203  0.240351  0.940617 -0.664254 -0.599859 -0.140781  0.080944   
487  0.653194  0.268082  0.447387 -0.735508 -0.328048 -0.057711  0.090916   

          7         8         9    ...       758       759       760  \
0  

In [7]:
X2_all.df["SMILES"] = X2_all.df["Drug"]
extractor2 = MolEmb.EmbeddingExtractor(model_name=model_name, df=X2_all.df)
emb2, X2_all.df = extractor2.get_embeddings()
print(emb2)


Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


          0         1         2         3         4         5         6    \
0    0.624928  0.242250  0.854679 -0.867963 -0.678615  0.090217 -0.139854   
1    0.374832  0.298939  0.985079 -0.960167 -0.591621  0.100837  0.483177   
2    0.431647  0.437582  0.698591 -0.589104 -0.381413 -0.372015  0.105332   
3    0.208212  0.726000  0.748835 -0.645130 -0.163467 -0.498499 -0.233751   
4    0.726675 -0.141403  0.820797 -0.664847 -0.127350 -0.111642 -0.066923   
..        ...       ...       ...       ...       ...       ...       ...   
150  0.087181  0.772850  0.894351 -0.344389 -0.262052 -0.413850 -0.225824   
151  0.600678  0.260839  0.673289 -0.940044 -0.124541  0.322733 -0.055883   
152  0.365983 -0.121044  1.093334 -0.537752 -0.664956  0.099662  0.120994   
153  0.325634  0.358863  0.697016 -0.559271 -0.433622 -0.195402  0.131260   
154  0.256436  0.172304  0.093762 -0.253850  0.268059 -0.568010 -0.210785   

          7         8         9    ...       758       759       760  \
0  

In [8]:
X3_all.df["SMILES"] = X3_all.df["Drug"]
extractor3 = MolEmb.EmbeddingExtractor(model_name=model_name, df=X3_all.df)
emb3, X3_all.df = extractor3.get_embeddings()
print(emb3)


Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


          0         1         2         3         4         5         6    \
0    0.279015  0.172189  0.993353 -0.794624 -0.272241  0.118495  0.194195   
1    0.304655  0.057044  0.914615 -0.915091 -0.532769  0.068886  0.277302   
2    0.672505  0.671482  0.135650 -0.781623 -0.451101 -0.612638  0.023349   
3    0.275461  0.193600  1.067724 -0.826211 -0.143357  0.060240  0.260925   
4    0.095226  0.187408  0.568077 -0.693950 -0.104264 -0.026288 -0.006962   
..        ...       ...       ...       ...       ...       ...       ...   
159  0.781352  0.516321  0.880270 -0.814452 -0.820508 -0.068080  0.047210   
160  0.485748  0.336550  0.648807 -0.620136 -0.627679 -0.359975 -0.102899   
161  0.522189  0.404734  0.003024 -0.593466 -0.324957 -0.540067 -0.465106   
162  0.468361  0.274435 -0.004998 -0.569882  0.013079 -0.137342 -0.440621   
163  0.803913  0.319314  0.522068 -0.561394 -0.389965 -0.444372 -0.243592   

          7         8         9    ...       758       759       760  \
0  

In [9]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
emb = emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, emb], axis = 1)

In [10]:
display(X1_all.X)

,MorganFP_MorganFP_0,MorganFP_MorganFP_1,MorganFP_MorganFP_2,MorganFP_MorganFP_3,MorganFP_MorganFP_4,MorganFP_MorganFP_5,MorganFP_MorganFP_6,MorganFP_MorganFP_7,MorganFP_MorganFP_8,MorganFP_MorganFP_9,...,758,759,760,761,762,763,764,765,766,767
0,False,False,False,False,False,False,False,False,False,False,...,-0.143628,0.515095,-0.074862,-0.102673,0.103030,0.130476,0.499113,-0.214322,-0.077564,-0.431823
1,False,False,False,False,False,False,False,False,False,False,...,0.051274,0.277398,-0.593216,0.042848,-0.011813,0.098258,0.729520,-0.701225,0.259042,-0.592370
2,False,True,False,False,False,False,False,False,False,False,...,-0.086957,0.134045,0.056459,-0.335765,0.136977,0.272289,1.180151,-0.274104,0.327263,-0.534128
3,False,False,False,False,False,False,False,False,False,False,...,-0.250937,0.206534,-0.223312,0.343322,-0.241193,0.214450,1.021452,-0.288710,-0.215809,-0.786897
4,False,False,False,True,False,False,False,False,False,False,...,0.054637,0.027199,-0.383653,0.068650,0.585950,-0.025223,0.926432,0.213663,-0.226463,-0.451652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,False,False,False,False,False,False,False,False,False,False,...,0.259191,-0.156011,-0.657177,0.090683,-0.131137,-0.044086,0.658195,-0.148066,-0.333033,-0.870234
484,False,False,False,False,False,False,False,False,False,False,...,0.279238,0.092157,-0.210265,-0.054687,0.159311,0.148841,1.103577,-0.306556,0.361056,-0.596046
485,False,False,False,False,False,False,False,False,True,False,...,-0.088278,0.292331,-0.240328,-0.308141,-0.375836,-0.005479,0.483670,-0.645472,0.012842,-0.240922
486,False,False,False,False,False,False,False,False,False,False,...,0.253126,0.356655,-0.030671,-0.108964,0.162669,0.056533,0.501798,-0.532637,-0.328418,-0.128803


In [11]:
X2_all.X = X2_all.X.reset_index(drop=True)
emb2 = emb2.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, emb2], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
emb3 = emb3.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, emb3], axis = 1)

In [12]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [14]:
X1.columns = X1.columns.astype(str)
X2.columns = X2.columns.astype(str)
X3.columns = X3.columns.astype(str)

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [15]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001
0,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.867205,1.562896,0.740015,-0.007128,0.302925,0.290817,-1.193038,0.389139,-0.488945,0.195620
1,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.010128,0.448418,-1.361342,0.603047,-0.248245,0.183608,-0.284350,-1.654110,0.848054,-0.329638
2,-0.090909,2.722828,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.617997,-0.223713,1.272376,-0.984487,0.465849,0.762727,1.492864,0.138268,1.119030,-0.139089
3,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-1.339098,0.116163,0.138212,1.862946,-1.349126,0.570258,0.866983,0.076975,-1.038054,-0.966066
4,-0.090909,-0.367265,-0.090909,4.053217,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,0.004662,-0.724673,-0.511796,0.711238,2.620634,-0.227298,0.492240,2.185144,-1.080374,0.130745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,0.718201,-1.459092,-0.386883,-1.513504,-1.252674,-2.065734,1.243633,0.522379,0.473160,0.724154
591,3.466833,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,0.801147,-1.028012,-0.336019,-1.439344,-1.121564,-1.984826,1.271150,0.336867,0.342815,0.574947
592,-0.090909,0.209094,-0.090909,0.555300,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.263113,0.870566,0.721030,-0.434983,-0.480444,0.155048,0.349936,-1.240762,-0.779169,0.998289
593,-0.090909,-0.367265,-0.090909,-0.246718,0.0,6.992298,0.0,-0.090909,-0.251358,-0.111571,...,-1.788570,-0.132143,-0.171981,-1.256847,-0.091292,0.619600,-1.287512,0.745633,-0.119099,0.476873


In [17]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [18]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    return my_df

In [19]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
import torch
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix
def objective(trial, X_train, y_train, X_test, y_test):
    dropout_frac = trial.suggest_categorical("dropout_frac", [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9])
    patience = trial.suggest_categorical("patience", [10, 40, 75])
    tol = trial.suggest_categorical("tol", [1e-5, 1e-4, 1e-3, 1e-2, 0])
    weight_decay = trial.suggest_categorical("weight_decay", [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0])
    n_epochs = trial.suggest_categorical("n_epochs", [200, 300, 500, 1000])
    batch_size = trial.suggest_categorical("batch_size", [1024, 512, 256, 128, 64])
    optimizer = trial.suggest_categorical("optimizer", ["optim.AdamW", "optim.RMSprop"])
    lr = trial.suggest_categorical("lr", [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
    neuron_layers_dict = {
    '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 1024, 256, 64, 8]': [4096, 1024, 256, 64, 8],
    '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]': [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096]': [4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096],
    '[200]': [200],
    '[2000]': [2000],
    '[2000, 1000]': [2000, 1000],
    '[2000, 1000, 500]': [2000, 1000, 500],
    '[1000, 50]': [1000, 50],
    '[4000, 2000]': [4000, 2000],
    '[4000, 2000, 1000, 500]': [4000, 2000, 1000, 500],
    '[4000, 2000, 2000, 500]': [4000, 2000, 2000, 500]
    }
    neuron_layers_size = trial.suggest_categorical("neuron_layers_size", list(neuron_layers_dict.keys()))
    opt = {"optim.AdamW": optim.AdamW,
           "optim.RMSprop": optim.RMSprop}
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    
    # Model
    model = STFullyConnected(
        n_dim=X_train.shape[1],
        n_class=1,
        gpus=[],
        device=device,
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=dropout_frac,
        patience=patience,
        tol=tol,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=weight_decay,
        n_epochs=n_epochs,
        neuron_layers= neuron_layers_dict[neuron_layers_size],  # Použití neuron_layers_size
        batch_size=batch_size,
        optimizer=opt[optimizer],
        lr=lr,
        random_seed=69
    )
    # Trénink a predikce
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_bin = preds > 0.5

    # Metiky
    f1 = f1_score(y_test, preds_bin)
    acc = accuracy_score(y_test, preds_bin)
    mcc = matthews_corrcoef(y_test, preds_bin)
    
    # Můžeš logovat i do trialu
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("acc", acc)
    display(confusion_matrix(y_test, preds_bin))
    return mcc  # maximalizujeme MCC


In [ ]:
study_3 = optuna.create_study(
    study_name="CK1_new_roberta",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.NSGAIISampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=200
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

[I 2025-05-03 15:31:30,839] A new study created in RDB with name: CK1_new_roberta


cuda


array([[125,   0],
       [ 30,   0]])

[I 2025-05-03 15:32:03,340] Trial 0 finished with value: 0.0 and parameters: {'dropout_frac': 0.4, 'patience': 40, 'tol': 1e-05, 'weight_decay': 0.1, 'n_epochs': 500, 'batch_size': 512, 'optimizer': 'optim.RMSprop', 'lr': 0.001, 'neuron_layers_size': '[2000, 1000, 500]'}. Best is trial 0 with value: 0.0.


cuda


...

[I 2025-05-03 16:10:22,339] Trial 100 finished with value: 0.42229029405283497 and parameters: {'dropout_frac': 0.6, 'patience': 10, 'tol': 0.01, 'weight_decay': 0.01, 'n_epochs': 300, 'batch_size': 256, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096]'}. Best is trial 21 with value: 0.4740706191139998.


cuda


array([[107,  18],
       [ 13,  17]])

[I 2025-05-03 16:10:42,768] Trial 101 finished with value: 0.39938245981308346 and parameters: {'dropout_frac': 0, 'patience': 40, 'tol': 0.001, 'weight_decay': 1e-05, 'n_epochs': 500, 'batch_size': 256, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[4096, 1024, 256, 64, 8]'}. Best is trial 21 with value: 0.4740706191139998.


cuda
